# json2openapi-llm — Ollama evaluation (Llama 3) on Google Colab

Evaluates **few_shot**, **cot** and **cove** prompting techniques on all **100** test samples.

**Before running:** `Runtime → Change runtime type → T4 GPU` (or better), then `Runtime → Run all`.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install Ollama + Python dependencies

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install requests pyyaml openapi-schema-validator

## 3. Start the Ollama server in the background and pull `llama3`

In [ ]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2)
        print("Ollama server is up.")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start - check /content/ollama.log")

In [ ]:
!ollama pull llama3

## 4. Write the evaluation script and the test dataset

In [ ]:
%%writefile /content/evaluate_ollama.py
import argparse
import json
import os
import re
import time
import requests
import yaml
from openapi_schema_validator import OAS30Validator

OLLAMA_API_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3"


def call_ollama(prompt: str) -> str:
    resp = requests.post(
        OLLAMA_API_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.0},
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["response"].strip()


def extract_yaml(raw_text: str) -> str:
    """Extracts clean YAML from the model response (after the FINAL_YAML: marker or inside ```yaml code fences)."""
    text = raw_text.strip()

    # 1. If markdown fences exist, take the LAST one (CoT / CoVe)
    fences = re.findall(r"```(?:yaml|yml)?\s*\n(.*?)```", text, re.DOTALL | re.IGNORECASE)
    if fences:
        return fences[-1].strip()

    # 2. Fallback if no markdown fences are present
    if "FINAL_YAML:" in text:
        text = text.split("FINAL_YAML:")[-1].strip()

    # Match 'type: object' at column 0 only (schema root, never a nested object)
    matches = list(re.finditer(r"^type:\s*object\b", text, re.MULTILINE))
    if matches:
        text = text[matches[-1].start():]

    return text.strip()
    


# --- TECHNIQUE 1: Few-Shot Prompting (Taxonomy: New Tasks Without Extensive Training) ---
def run_few_shot(input_json: str) -> str:
    prompt = f"""You are an expert backend API architect. Generate a valid OpenAPI 3.0 YAML schema for the provided JSON.
Rules:
- Return ONLY raw YAML. No explanations, no markdown fences.
- Map null values to: type: string with nullable: true.
- Map empty arrays to: type: array with items: {{}}.
- Use integer for whole numbers and number for floats.

EXAMPLE 1 INPUT:
{{"status": "ok", "count": 5, "error": null, "tags": []}}

EXAMPLE 1 OUTPUT:
type: object
properties:
  status:
    type: string
  count:
    type: integer
  error:
    type: string
    nullable: true
  tags:
    type: array
    items: {{}}

EXAMPLE 2 INPUT:
{{"service": "auth", "roles": ["admin"], "meta": {{"active": true, "latency": 12.5}}, "items": [{{"id": 1}}]}}

EXAMPLE 2 OUTPUT:
type: object
properties:
  service:
    type: string
  roles:
    type: array
    items:
      type: string
  meta:
    type: object
    properties:
      active:
        type: boolean
      latency:
        type: number
  items:
    type: array
    items:
      type: object
      properties:
        id:
          type: integer

ACTUAL INPUT:
{input_json}

ACTUAL OUTPUT:"""
    return extract_yaml(call_ollama(prompt))


# --- TECHNIQUE 2: Chain-of-Thought (Taxonomy: Reasoning and Logic) ---
def run_cot(input_json: str) -> str:
    prompt = f"""You are an expert backend API architect. Convert the following JSON object into a valid OpenAPI 3.0 YAML Schema Object.

Strict formatting rules:
- Output ONLY the Schema Object starting directly with 'type: object' and 'properties:'.
- Do NOT generate a full OpenAPI document (do NOT include 'openapi:', 'info:', 'paths:', or 'components:').
- Do NOT add extra fields like 'required:', 'additionalProperties:', 'description:', or 'example:'.
- Map null values to: 'type: string' with 'nullable: true'.
- Map empty arrays [] to:
  type: array
  items: {{}}
- Map non-empty string arrays ["a", "b"] to:
  type: array
  items:
    type: string
  (NEVER write 'items: string' or 'items: type: string' on the same line!)
- Map non-empty object arrays [{{...}}] to 'type: array' with 'items:' containing 'type: object' and 'properties:'.
- Use 'integer' for whole numbers and 'number' for decimals.

Think step by step:
1. List every top-level key and nested key in the JSON with its exact OpenAPI 3.0 type.
2. Check special cases (null values, empty arrays [], non-empty arrays).
3. Keep your reasoning concise (max 5 bullet points), then output the final YAML Schema Object inside a single ```yaml code block starting with 'type: object'.

JSON INPUT:
{input_json}

Let's think step by step:"""
    return extract_yaml(call_ollama(prompt))


# --- TECHNIQUE 3: Chain-of-Verification / CoVe (Taxonomy: Reduce Hallucination) ---
def run_cove(input_json: str) -> str:
    # Step 1: Generate initial draft
    draft_prompt = f"""You are an expert backend API architect. Generate an OpenAPI 3.0 YAML Schema Object for this JSON.
Rules:
- Start directly with 'type: object' and 'properties:'.
- Do NOT include 'openapi:', 'info:', 'paths:', 'components:', 'required:', or 'example:'.
- Map null values to: 'type: string' with 'nullable: true'.
- Map empty arrays [] to: 'type: array' with 'items: {{}}'.
- Map non-empty string arrays ["a"] to:
  type: array
  items:
    type: string
- Use 'integer' for whole numbers and 'number' for decimals.
- Return ONLY raw YAML inside a ```yaml code block.

JSON:
{input_json}

YAML:"""
    draft_yaml = extract_yaml(call_ollama(draft_prompt))

    # Step 2: Verify draft and produce corrected output
    verify_prompt = f"""You are an OpenAPI 3.0 schema auditor. Verify the DRAFT YAML against the ORIGINAL JSON and fix ONLY actual errors.

ORIGINAL JSON:
{input_json}

DRAFT YAML:
{draft_yaml}

Verification checklist:
1. Exact keys: Does DRAFT YAML contain the exact same keys as ORIGINAL JSON at the same nesting levels? Do NOT invent or add any extra keys (such as 'nullable' or 'required') that are not in ORIGINAL JSON!
2. Numbers: Are whole numbers typed as 'integer' and decimals typed as 'number'?
3. Null values (ONLY if ORIGINAL JSON contains a null value): Is that specific key typed as 'type: string' with 'nullable: true'?
4. Empty arrays (ONLY if ORIGINAL JSON contains []): Does that key use 'type: array' and 'items: {{}}'? Do NOT change 'items: {{}}' to 'type: string' or 'type: object' if the JSON array is empty `[]`!
5. Non-empty arrays (ONLY if ORIGINAL JSON contains elements like ["a", "b"] or [{{...}}]): Does 'items:' have 'type: string' (for string arrays) or 'type: object' (for object arrays) indented on the next line? (Never write 'items: string' or 'items: type: string' on one line).

Briefly answer the 5 checks (max 5 short lines). If DRAFT YAML is already correct, keep it unchanged. Output the final OpenAPI 3.0 YAML Schema Object inside a single ```yaml code block starting with 'type: object'."""
    return extract_yaml(call_ollama(verify_prompt))


def evaluate_prediction(input_json_str: str, target_yaml_str: str, pred_yaml_str: str) -> dict:
    input_obj = json.loads(input_json_str)
    target_schema = yaml.safe_load(target_yaml_str)

    yaml_valid = False
    oas_valid = False
    json_valid = False
    exact_match = False
    error_msg = None

    try:
        pred_schema = yaml.safe_load(pred_yaml_str)
        if not isinstance(pred_schema, dict):
            raise ValueError("Parsed YAML is not a dictionary")
        yaml_valid = True

        if pred_schema.get("type") != "object" or "properties" not in pred_schema:
            raise ValueError("Root schema must be an OpenAPI Schema Object with 'type: object' and 'properties'")

        OAS30Validator.check_schema(pred_schema)
        oas_valid = True

        OAS30Validator(pred_schema).validate(input_obj)
        json_valid = True

        exact_match = pred_schema == target_schema
    except Exception as e:
        error_msg = str(e)[:200]

    return {
        "yaml_valid": yaml_valid,
        "oas_valid": oas_valid,
        "json_valid": json_valid,
        "exact_match": exact_match,
        "error": error_msg,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--limit", type=int, default=100, help="Number of test samples to evaluate")
    args = parser.parse_args()

    with open("data/test.jsonl", "r", encoding="utf-8") as f:
        test_data = [json.loads(line) for line in f][: args.limit]

    techniques = {
        "few_shot": run_few_shot,
        "cot": run_cot,
        "cove": run_cove,
    }

    os.makedirs("results", exist_ok=True)
    summary = {}

    for name, fn in techniques.items():
        print(f"\n=== Evaluating technique: {name} ({len(test_data)} samples) ===")
        records = []
        t0_all = time.time()

        for i, row in enumerate(test_data, 1):
            t0 = time.time()
            pred_yaml = fn(row["input_json"])
            elapsed = round(time.time() - t0, 2)

            metrics = evaluate_prediction(row["input_json"], row["target_yaml"], pred_yaml)
            records.append({
                "index": i,
                "latency_s": elapsed,
                "pred_yaml": pred_yaml,
                **metrics,
            })

            if i % 5 == 0 or i == len(test_data):
                print(
                    f"  [{i}/{len(test_data)}] "
                    f"OAS_valid={sum(r['oas_valid'] for r in records)}/{i}, "
                    f"JSON_valid={sum(r['json_valid'] for r in records)}/{i}, "
                    f"Exact={sum(r['exact_match'] for r in records)}/{i}"
                )

        n = len(records)
        stats = {
            "count": n,
            "yaml_parse_rate": round(100 * sum(r["yaml_valid"] for r in records) / n, 2),
            "oas_schema_valid_rate": round(100 * sum(r["oas_valid"] for r in records) / n, 2),
            "json_validation_accuracy": round(100 * sum(r["json_valid"] for r in records) / n, 2),
            "exact_match_rate": round(100 * sum(r["exact_match"] for r in records) / n, 2),
            "avg_latency_s": round(sum(r["latency_s"] for r in records) / n, 2),
            "total_time_s": round(time.time() - t0_all, 2),
        }
        summary[name] = stats

        with open(f"results/ollama_{name}.json", "w", encoding="utf-8") as out:
            json.dump({"stats": stats, "records": records}, out, indent=2, ensure_ascii=False)

    with open("results/ollama_summary.json", "w", encoding="utf-8") as out:
        json.dump(summary, out, indent=2, ensure_ascii=False)

    print("\n=== FINAL SUMMARY (OLLAMA) ===")
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
!mkdir -p /content/data

In [ ]:
%%writefile /content/data/test.jsonl
{"input_json": "{\"status\": \"ok\", \"data\": {\"endpoint\": \"hub.load('https://tfhub.dev/google/unive...\", \"score\": 0.59, \"framework\": \"TensorFlow Hub\", \"provider\": \"TensorFlow Hub\", \"latency_ms\": 393.42, \"arg_count\": 1}, \"meta\": {\"version\": 3, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      endpoint:\n        type: string\n      score:\n        type: number\n      framework:\n        type: string\n      provider:\n        type: string\n      latency_ms:\n        type: number\n      arg_count:\n        type: integer\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"total\": 1, \"entries\": [{\"name\": \"glpn-kitti-finetuned-diode\", \"enabled\": true, \"weight\": 0.41}], \"details\": {\"id\": 25360, \"deprecated\": true, \"active\": false, \"endpoint\": \"AutoModel.from_pretrained('sayakpaul/glp...\", \"score\": 0.551}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      id:\n        type: integer\n      deprecated:\n        type: boolean\n      active:\n        type: boolean\n      endpoint:\n        type: string\n      score:\n        type: number"}
{"input_json": "{\"domain\": \"Computer Vision Video Cla...\", \"framework\": \"Hugging Face Transformers\", \"tags\": [\"Computer Vision Video Cla...\", \"Hugging Face Transformers\"], \"id\": 86704}", "target_yaml": "type: object\nproperties:\n  domain:\n    type: string\n  framework:\n    type: string\n  tags:\n    type: array\n    items:\n      type: string\n  id:\n    type: integer"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"score\": 0.623, \"endpoint\": \"SequenceTagger.load('flair/ner-english-l...\", \"domain\": \"Natural Language Processi...\"}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  score:\n    type: number\n  endpoint:\n    type: string\n  domain:\n    type: string"}
{"input_json": "{\"service\": \"imagenet_mobilenet_v2_100_224_...\", \"config\": {\"error\": null, \"deprecated\": false, \"arguments\": [], \"arg_count\": 0, \"requirements\": []}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      error:\n        type: string\n        nullable: true\n      deprecated:\n        type: boolean\n      arguments:\n        type: array\n        items: {}\n      arg_count:\n        type: integer\n      requirements:\n        type: array\n        items: {}\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"tags\": [], \"status_code\": 202, \"latency_ms\": 281.77, \"framework\": \"Hugging Face Transformers\"}, \"meta\": {\"version\": 3, \"cached\": false}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      tags:\n        type: array\n        items: {}\n      status_code:\n        type: integer\n      latency_ms:\n        type: number\n      framework:\n        type: string\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"service\": \"imagenet-mobilenet_v2_100_224-...\", \"config\": {\"requirements\": [], \"tags\": [\"Image feature vector\", \"TensorFlow Hub\"], \"arg_count\": 0, \"status_code\": 404, \"active\": false, \"deprecated\": true}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      requirements:\n        type: array\n        items: {}\n      tags:\n        type: array\n        items:\n          type: string\n      arg_count:\n        type: integer\n      status_code:\n        type: integer\n      active:\n        type: boolean\n      deprecated:\n        type: boolean\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"api_name\": \"Antheia/Hanna\", \"arg_count\": 0, \"deprecated\": true, \"requirements\": []}, \"meta\": {\"version\": 1, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      api_name:\n        type: string\n      arg_count:\n        type: integer\n      deprecated:\n        type: boolean\n      requirements:\n        type: array\n        items: {}\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"provider\": \"Hugging Face\", \"tags\": [], \"requirements\": [\"transformers\"]}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  tags:\n    type: array\n    items: {}\n  requirements:\n    type: array\n    items:\n      type: string"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"provider\": \"TensorFlow Hub\", \"status_code\": 202, \"error\": null, \"latency_ms\": 37.13, \"domain\": \"Image feature vector\"}, \"meta\": {\"version\": 3, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      provider:\n        type: string\n      status_code:\n        type: integer\n      error:\n        type: string\n        nullable: true\n      latency_ms:\n        type: number\n      domain:\n        type: string\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"service\": \"facebook/tts_transformer-fr-cv...\", \"config\": {\"id\": 75530, \"framework\": \"Fairseq\", \"status_code\": 200, \"requirements\": [\"fairseq\"]}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      id:\n        type: integer\n      framework:\n        type: string\n      status_code:\n        type: integer\n      requirements:\n        type: array\n        items:\n          type: string\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"service\": \"xm_transformer_s2ut_en-hk\", \"config\": {\"latency_ms\": 247.85, \"domain\": \"Audio Audio-to-Audio\", \"framework\": \"Fairseq\", \"arg_count\": 0}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      latency_ms:\n        type: number\n      domain:\n        type: string\n      framework:\n        type: string\n      arg_count:\n        type: integer\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"status_code\": 202, \"active\": true, \"endpoint\": \"hub.load('https://tfhub.dev/tensorflow/s...\", \"score\": 0.707, \"domain\": \"Image object detection\"}", "target_yaml": "type: object\nproperties:\n  status_code:\n    type: integer\n  active:\n    type: boolean\n  endpoint:\n    type: string\n  score:\n    type: number\n  domain:\n    type: string"}
{"input_json": "{\"status_code\": 404, \"tags\": [\"Computer Vision Depth Est...\", \"Hugging Face Transformers\"], \"id\": 81796, \"provider\": \"Hugging Face Transformers\"}", "target_yaml": "type: object\nproperties:\n  status_code:\n    type: integer\n  tags:\n    type: array\n    items:\n      type: string\n  id:\n    type: integer\n  provider:\n    type: string"}
{"input_json": "{\"service\": \"blip-image-captioning-large\", \"config\": {\"api_name\": \"blip-image-captioning-large\", \"framework\": \"Transformers\", \"error\": null, \"score\": 0.669, \"latency_ms\": 72.58, \"deprecated\": true}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      api_name:\n        type: string\n      framework:\n        type: string\n      error:\n        type: string\n        nullable: true\n      score:\n        type: number\n      latency_ms:\n        type: number\n      deprecated:\n        type: boolean\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"framework\": \"TensorFlow Hub\", \"latency_ms\": 397.87, \"active\": false}, \"meta\": {\"version\": 2, \"cached\": false}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      framework:\n        type: string\n      latency_ms:\n        type: number\n      active:\n        type: boolean\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"requirements\": [], \"score\": 0.565, \"provider\": \"Transformers\"}", "target_yaml": "type: object\nproperties:\n  requirements:\n    type: array\n    items: {}\n  score:\n    type: number\n  provider:\n    type: string"}
{"input_json": "{\"service\": \"rasa/LaBSE\", \"config\": {\"id\": 4139, \"latency_ms\": 316.61, \"arg_count\": 0, \"arguments\": [], \"framework\": \"Hugging Face Transformers\", \"status_code\": 400}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      id:\n        type: integer\n      latency_ms:\n        type: number\n      arg_count:\n        type: integer\n      arguments:\n        type: array\n        items: {}\n      framework:\n        type: string\n      status_code:\n        type: integer\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"api_name\": \"universal-sentence-encoder\", \"deprecated\": true, \"endpoint\": \"hub.load('https://tfhub.dev/google/unive...\", \"arg_count\": 1, \"domain\": \"Text embedding\"}", "target_yaml": "type: object\nproperties:\n  api_name:\n    type: string\n  deprecated:\n    type: boolean\n  endpoint:\n    type: string\n  arg_count:\n    type: integer\n  domain:\n    type: string"}
{"input_json": "{\"provider\": \"pyannote.audio\", \"requirements\": [], \"arg_count\": 3, \"active\": false, \"score\": 0.927, \"framework\": \"pyannote.audio\"}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  requirements:\n    type: array\n    items: {}\n  arg_count:\n    type: integer\n  active:\n    type: boolean\n  score:\n    type: number\n  framework:\n    type: string"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"endpoint\": \"hub.load('https://tfhub.dev/google/unive...\", \"domain\": \"Text embedding\", \"active\": true, \"api_name\": \"universal-sentence-encoder\", \"id\": 59873}, \"meta\": {\"version\": 2, \"cached\": false}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      endpoint:\n        type: string\n      domain:\n        type: string\n      active:\n        type: boolean\n      api_name:\n        type: string\n      id:\n        type: integer\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"service\": \"universal-sentence-encoder\", \"config\": {\"endpoint\": \"hub.load('https://tfhub.dev/google/unive...\", \"deprecated\": true, \"framework\": \"TensorFlow Hub\", \"requirements\": [], \"active\": false}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      endpoint:\n        type: string\n      deprecated:\n        type: boolean\n      framework:\n        type: string\n      requirements:\n        type: array\n        items: {}\n      active:\n        type: boolean\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"service\": \"code_trans_t5_base_code_docume...\", \"config\": {\"tags\": [], \"arg_count\": 1, \"domain\": \"Natural Language Processi...\", \"latency_ms\": 289.67}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      tags:\n        type: array\n        items: {}\n      arg_count:\n        type: integer\n      domain:\n        type: string\n      latency_ms:\n        type: number\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"arguments\": [], \"active\": false, \"framework\": \"Hugging Face Transformers\", \"api_name\": \"DCUNet_Libri1Mix_enhsingle_16k\"}", "target_yaml": "type: object\nproperties:\n  arguments:\n    type: array\n    items: {}\n  active:\n    type: boolean\n  framework:\n    type: string\n  api_name:\n    type: string"}
{"input_json": "{\"service\": \"lllyasviel/control_v11p_sd15_o...\", \"config\": {\"error\": null, \"arg_count\": 0, \"score\": 0.891}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      error:\n        type: string\n        nullable: true\n      arg_count:\n        type: integer\n      score:\n        type: number\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"arguments\": [], \"id\": 85402, \"arg_count\": 1}", "target_yaml": "type: object\nproperties:\n  arguments:\n    type: array\n    items: {}\n  id:\n    type: integer\n  arg_count:\n    type: integer"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"arg_count\": 0, \"score\": 0.704, \"status_code\": 202, \"api_name\": \"hkunlp/instructor-base\", \"arguments\": []}, \"meta\": {\"version\": 1, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      arg_count:\n        type: integer\n      score:\n        type: number\n      status_code:\n        type: integer\n      api_name:\n        type: string\n      arguments:\n        type: array\n        items: {}\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"arg_count\": 0, \"tags\": [\"Text embedding\", \"TensorFlow Hub\"], \"requirements\": [], \"status_code\": 200, \"deprecated\": true}", "target_yaml": "type: object\nproperties:\n  arg_count:\n    type: integer\n  tags:\n    type: array\n    items:\n      type: string\n  requirements:\n    type: array\n    items: {}\n  status_code:\n    type: integer\n  deprecated:\n    type: boolean"}
{"input_json": "{\"score\": 0.665, \"domain\": \"Audio Text-to-Speech\", \"arguments\": [], \"error\": null, \"deprecated\": false, \"active\": false}", "target_yaml": "type: object\nproperties:\n  score:\n    type: number\n  domain:\n    type: string\n  arguments:\n    type: array\n    items: {}\n  error:\n    type: string\n    nullable: true\n  deprecated:\n    type: boolean\n  active:\n    type: boolean"}
{"input_json": "{\"requirements\": [], \"score\": 0.719, \"deprecated\": true, \"framework\": \"Hugging Face Transformers\", \"latency_ms\": 311.96, \"error\": null}", "target_yaml": "type: object\nproperties:\n  requirements:\n    type: array\n    items: {}\n  score:\n    type: number\n  deprecated:\n    type: boolean\n  framework:\n    type: string\n  latency_ms:\n    type: number\n  error:\n    type: string\n    nullable: true"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"total\": 1, \"entries\": [{\"name\": \"Helsinki-NLP/opus-mt-zh-en\", \"enabled\": true, \"weight\": 0.73}], \"details\": {\"tags\": [\"Natural Language Processi...\", \"Hugging Face Transformers\"], \"framework\": \"Hugging Face Transformers\", \"requirements\": [], \"domain\": \"Natural Language Processi...\", \"error\": null}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      tags:\n        type: array\n        items:\n          type: string\n      framework:\n        type: string\n      requirements:\n        type: array\n        items: {}\n      domain:\n        type: string\n      error:\n        type: string\n        nullable: true"}
{"input_json": "{\"arg_count\": 1, \"tags\": [\"Natural Language Processi...\", \"Hugging Face Transformers\"], \"latency_ms\": 387.89, \"endpoint\": \"BartModel.from_pretrained('facebook/bart...\"}", "target_yaml": "type: object\nproperties:\n  arg_count:\n    type: integer\n  tags:\n    type: array\n    items:\n      type: string\n  latency_ms:\n    type: number\n  endpoint:\n    type: string"}
{"input_json": "{\"arguments\": [], \"error\": null, \"api_name\": \"model_id\", \"endpoint\": \"hub.load('https://tfhub.dev/google/yamne...\", \"arg_count\": 1}", "target_yaml": "type: object\nproperties:\n  arguments:\n    type: array\n    items: {}\n  error:\n    type: string\n    nullable: true\n  api_name:\n    type: string\n  endpoint:\n    type: string\n  arg_count:\n    type: integer"}
{"input_json": "{\"latency_ms\": 308.65, \"deprecated\": false, \"status_code\": 202, \"api_name\": \"model_id\", \"domain\": \"Image classification\"}", "target_yaml": "type: object\nproperties:\n  latency_ms:\n    type: number\n  deprecated:\n    type: boolean\n  status_code:\n    type: integer\n  api_name:\n    type: string\n  domain:\n    type: string"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"latency_ms\": 294.5, \"deprecated\": true, \"framework\": \"Fairseq\", \"id\": 25609, \"api_name\": \"xm_transformer_s2ut_hk-en\", \"score\": 0.944}, \"meta\": {\"version\": 3, \"cached\": false}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      latency_ms:\n        type: number\n      deprecated:\n        type: boolean\n      framework:\n        type: string\n      id:\n        type: integer\n      api_name:\n        type: string\n      score:\n        type: number\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"service\": \"michellejieli/NSFW_text_classi...\", \"config\": {\"arguments\": [\"text\"], \"endpoint\": \"pipeline('sentiment-analysis', model='mi...\", \"provider\": \"Transformers\", \"score\": 0.823}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      arguments:\n        type: array\n        items:\n          type: string\n      endpoint:\n        type: string\n      provider:\n        type: string\n      score:\n        type: number\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"id\": 10389, \"domain\": \"Image classification\", \"endpoint\": \"hub.KerasLayer('https://tfhub.dev/google...\"}, \"meta\": {\"version\": 3, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      id:\n        type: integer\n      domain:\n        type: string\n      endpoint:\n        type: string\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"domain\": \"Natural Language Processi...\", \"deprecated\": false, \"requirements\": [], \"score\": 0.944}", "target_yaml": "type: object\nproperties:\n  domain:\n    type: string\n  deprecated:\n    type: boolean\n  requirements:\n    type: array\n    items: {}\n  score:\n    type: number"}
{"input_json": "{\"service\": \"universal-sentence-encoder\", \"config\": {\"arguments\": [\"input_text\"], \"deprecated\": true, \"status_code\": 201, \"domain\": \"Text embedding\", \"latency_ms\": 432.56}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      arguments:\n        type: array\n        items:\n          type: string\n      deprecated:\n        type: boolean\n      status_code:\n        type: integer\n      domain:\n        type: string\n      latency_ms:\n        type: number\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"active\": true, \"tags\": [\"Image feature vector\", \"TensorFlow Hub\"], \"error\": null, \"arg_count\": 0}", "target_yaml": "type: object\nproperties:\n  active:\n    type: boolean\n  tags:\n    type: array\n    items:\n      type: string\n  error:\n    type: string\n    nullable: true\n  arg_count:\n    type: integer"}
{"input_json": "{\"service\": \"srg/outhimar_64-Close-regressi...\", \"config\": {\"deprecated\": false, \"framework\": \"Scikit-learn\", \"provider\": \"Scikit-learn\", \"arg_count\": 2, \"requirements\": [\"scikit-learn\", \"dabl\"]}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      deprecated:\n        type: boolean\n      framework:\n        type: string\n      provider:\n        type: string\n      arg_count:\n        type: integer\n      requirements:\n        type: array\n        items:\n          type: string\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"service\": \"codet5-large-ntp-py\", \"config\": {\"active\": true, \"api_name\": \"codet5-large-ntp-py\", \"framework\": \"Hugging Face Transformers\", \"id\": 93794, \"provider\": \"Hugging Face Transformers\", \"deprecated\": true}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      active:\n        type: boolean\n      api_name:\n        type: string\n      framework:\n        type: string\n      id:\n        type: integer\n      provider:\n        type: string\n      deprecated:\n        type: boolean\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"service\": \"universal-sentence-encoder\", \"config\": {\"active\": false, \"status_code\": 201, \"provider\": \"TensorFlow Hub\", \"latency_ms\": 377.91, \"requirements\": [\"tensorflow\", \"tensorflow_hub\"]}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      active:\n        type: boolean\n      status_code:\n        type: integer\n      provider:\n        type: string\n      latency_ms:\n        type: number\n      requirements:\n        type: array\n        items:\n          type: string\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"domain\": \"Image classification\", \"arg_count\": 0, \"provider\": \"TensorFlow Hub\", \"requirements\": [], \"deprecated\": true, \"error\": null}", "target_yaml": "type: object\nproperties:\n  domain:\n    type: string\n  arg_count:\n    type: integer\n  provider:\n    type: string\n  requirements:\n    type: array\n    items: {}\n  deprecated:\n    type: boolean\n  error:\n    type: string\n    nullable: true"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"provider\": \"TensorFlow Hub\", \"latency_ms\": 72.86, \"error\": null, \"requirements\": []}, \"meta\": {\"version\": 1, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      provider:\n        type: string\n      latency_ms:\n        type: number\n      error:\n        type: string\n        nullable: true\n      requirements:\n        type: array\n        items: {}\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"score\": 0.946, \"arg_count\": 0, \"domain\": \"Multimodal Visual Questio...\", \"provider\": \"Hugging Face Transformers\", \"tags\": [\"Multimodal Visual Questio...\", \"Hugging Face Transformers\"], \"error\": null}, \"meta\": {\"version\": 1, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      score:\n        type: number\n      arg_count:\n        type: integer\n      domain:\n        type: string\n      provider:\n        type: string\n      tags:\n        type: array\n        items:\n          type: string\n      error:\n        type: string\n        nullable: true\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"requirements\": [], \"provider\": \"Transformers\", \"id\": 40315, \"domain\": \"Natural Language Processi...\"}, \"meta\": {\"version\": 1, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      requirements:\n        type: array\n        items: {}\n      provider:\n        type: string\n      id:\n        type: integer\n      domain:\n        type: string\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"tags\": [\"Text embedding\", \"TensorFlow Hub\"], \"endpoint\": \"hub.load('https://tfhub.dev/google/unive...\", \"latency_ms\": 390.85}, \"meta\": {\"version\": 1, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      tags:\n        type: array\n        items:\n          type: string\n      endpoint:\n        type: string\n      latency_ms:\n        type: number\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"service\": \"keras-io/tab_transformer\", \"config\": {\"domain\": \"Tabular Tabular Classific...\", \"active\": true, \"id\": 11615}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      domain:\n        type: string\n      active:\n        type: boolean\n      id:\n        type: integer\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"service\": \"madhurjindal/autonlp-Gibberish...\", \"config\": {\"framework\": \"Hugging Face Transformers\", \"latency_ms\": 47.96, \"arguments\": []}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      framework:\n        type: string\n      latency_ms:\n        type: number\n      arguments:\n        type: array\n        items: {}\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"tags\": [], \"domain\": \"Text embedding\", \"provider\": \"TensorFlow Hub\", \"score\": 0.641, \"endpoint\": \"hub.load('https://tfhub.dev/google/unive...\"}", "target_yaml": "type: object\nproperties:\n  tags:\n    type: array\n    items: {}\n  domain:\n    type: string\n  provider:\n    type: string\n  score:\n    type: number\n  endpoint:\n    type: string"}
{"input_json": "{\"service\": \"universal-sentence-encoder\", \"config\": {\"id\": 66823, \"error\": null, \"latency_ms\": 364.17}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      id:\n        type: integer\n      error:\n        type: string\n        nullable: true\n      latency_ms:\n        type: number\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"provider\": \"TensorFlow Hub\", \"status_code\": 201, \"active\": true, \"framework\": \"TensorFlow Hub\", \"api_name\": \"universal-sentence-encoder\"}, \"meta\": {\"version\": 2, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      provider:\n        type: string\n      status_code:\n        type: integer\n      active:\n        type: boolean\n      framework:\n        type: string\n      api_name:\n        type: string\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"active\": true, \"deprecated\": false, \"error\": null, \"id\": 9614, \"provider\": \"TensorFlow Hub\", \"endpoint\": \"hub.KerasLayer('https://tfhub.dev/google...\"}", "target_yaml": "type: object\nproperties:\n  active:\n    type: boolean\n  deprecated:\n    type: boolean\n  error:\n    type: string\n    nullable: true\n  id:\n    type: integer\n  provider:\n    type: string\n  endpoint:\n    type: string"}
{"input_json": "{\"provider\": \"TensorFlow Hub\", \"total\": 2, \"entries\": [{\"name\": \"universal-sentence-encoder\", \"enabled\": true, \"weight\": 0.4}], \"details\": {\"status_code\": 404, \"arg_count\": 1, \"latency_ms\": 137.01, \"domain\": \"Text preprocessing\"}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      status_code:\n        type: integer\n      arg_count:\n        type: integer\n      latency_ms:\n        type: number\n      domain:\n        type: string"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"total\": 1, \"entries\": [{\"name\": \"frizwankhan/entity-linking-mod...\", \"enabled\": true, \"weight\": 0.79}], \"details\": {\"error\": null, \"domain\": \"Multimodal Document Quest...\", \"arguments\": [], \"active\": false, \"arg_count\": 0, \"framework\": \"Hugging Face Transformers\"}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      error:\n        type: string\n        nullable: true\n      domain:\n        type: string\n      arguments:\n        type: array\n        items: {}\n      active:\n        type: boolean\n      arg_count:\n        type: integer\n      framework:\n        type: string"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"id\": 39473, \"domain\": \"Text embedding\", \"active\": false, \"api_name\": \"universal-sentence-encoder\", \"tags\": [\"Text embedding\", \"TensorFlow Hub\"]}, \"meta\": {\"version\": 3, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      id:\n        type: integer\n      domain:\n        type: string\n      active:\n        type: boolean\n      api_name:\n        type: string\n      tags:\n        type: array\n        items:\n          type: string\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"total\": 1, \"entries\": [{\"name\": \"microsoft/trocr-small-printed\", \"enabled\": true, \"weight\": 0.91}], \"details\": {\"latency_ms\": 57.11, \"arguments\": [], \"deprecated\": false}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      latency_ms:\n        type: number\n      arguments:\n        type: array\n        items: {}\n      deprecated:\n        type: boolean"}
{"input_json": "{\"domain\": \"Tabular Tabular Regressio...\", \"latency_ms\": 358.02, \"error\": null, \"provider\": \"Joblib\"}", "target_yaml": "type: object\nproperties:\n  domain:\n    type: string\n  latency_ms:\n    type: number\n  error:\n    type: string\n    nullable: true\n  provider:\n    type: string"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"domain\": \"Image object detection\", \"score\": 0.869, \"deprecated\": false, \"error\": null, \"arg_count\": 1}, \"meta\": {\"version\": 2, \"cached\": false}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      domain:\n        type: string\n      score:\n        type: number\n      deprecated:\n        type: boolean\n      error:\n        type: string\n        nullable: true\n      arg_count:\n        type: integer\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"service\": \"jwan2021/autotrain-us-housing-...\", \"config\": {\"arguments\": [\"data\"], \"requirements\": [\"joblib\", \"pandas\"], \"score\": 0.687, \"deprecated\": true}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      arguments:\n        type: array\n        items:\n          type: string\n      requirements:\n        type: array\n        items:\n          type: string\n      score:\n        type: number\n      deprecated:\n        type: boolean\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"error\": null, \"provider\": \"Hugging Face Transformers\", \"deprecated\": true, \"framework\": \"Hugging Face Transformers\", \"api_name\": \"typeform/squeezebert-mnli\"}, \"meta\": {\"version\": 3, \"cached\": false}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      error:\n        type: string\n        nullable: true\n      provider:\n        type: string\n      deprecated:\n        type: boolean\n      framework:\n        type: string\n      api_name:\n        type: string\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"latency_ms\": 299.51, \"arg_count\": 1, \"endpoint\": \"AutoModelForCausalLM.from_pretrained('fa...\", \"requirements\": [\"transformers\", \"torch\"], \"provider\": \"Hugging Face Transformers\"}, \"meta\": {\"version\": 2, \"cached\": false}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      latency_ms:\n        type: number\n      arg_count:\n        type: integer\n      endpoint:\n        type: string\n      requirements:\n        type: array\n        items:\n          type: string\n      provider:\n        type: string\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"arg_count\": 0, \"status_code\": 404, \"id\": 12130, \"active\": true, \"error\": null}, \"meta\": {\"version\": 1, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      arg_count:\n        type: integer\n      status_code:\n        type: integer\n      id:\n        type: integer\n      active:\n        type: boolean\n      error:\n        type: string\n        nullable: true\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"active\": false, \"error\": null, \"tags\": [\"Text preprocessing\", \"TensorFlow Hub\"], \"framework\": \"TensorFlow Hub\", \"id\": 86795}, \"meta\": {\"version\": 3, \"cached\": false}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      active:\n        type: boolean\n      error:\n        type: string\n        nullable: true\n      tags:\n        type: array\n        items:\n          type: string\n      framework:\n        type: string\n      id:\n        type: integer\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"total\": 1, \"entries\": [{\"name\": \"deepset/bert-base-cased-squad2\", \"enabled\": true, \"weight\": 0.4}], \"details\": {\"endpoint\": \"AutoModelForQuestionAnswering.from_pretr...\", \"arg_count\": 0, \"score\": 0.7, \"deprecated\": false, \"id\": 62541}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      endpoint:\n        type: string\n      arg_count:\n        type: integer\n      score:\n        type: number\n      deprecated:\n        type: boolean\n      id:\n        type: integer"}
{"input_json": "{\"service\": \"nitrosocke/nitro-diffusion\", \"config\": {\"arg_count\": 1, \"latency_ms\": 64.65, \"active\": true, \"tags\": [\"Multimodal Text-to-Image\", \"Hugging Face\"], \"provider\": \"Hugging Face\", \"requirements\": [\"torch\", \"diffusers\"]}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      arg_count:\n        type: integer\n      latency_ms:\n        type: number\n      active:\n        type: boolean\n      tags:\n        type: array\n        items:\n          type: string\n      provider:\n        type: string\n      requirements:\n        type: array\n        items:\n          type: string\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"service\": \"flair/ner-english-ontonotes-la...\", \"config\": {\"latency_ms\": 248.17, \"status_code\": 200, \"api_name\": \"flair/ner-english-ontonotes-la...\"}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      latency_ms:\n        type: number\n      status_code:\n        type: integer\n      api_name:\n        type: string\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"framework\": \"Hugging Face Transformers\", \"arguments\": [\"repo_id\", \"filename\"], \"score\": 0.629, \"arg_count\": 2}", "target_yaml": "type: object\nproperties:\n  framework:\n    type: string\n  arguments:\n    type: array\n    items:\n      type: string\n  score:\n    type: number\n  arg_count:\n    type: integer"}
{"input_json": "{\"provider\": \"TensorFlow Hub\", \"total\": 1, \"entries\": [{\"name\": \"imagenet_mobilenet_v2_130_224_...\", \"enabled\": true, \"weight\": 0.37}], \"details\": {\"domain\": \"Image classification\", \"api_name\": \"imagenet_mobilenet_v2_130_224_...\", \"error\": null}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      domain:\n        type: string\n      api_name:\n        type: string\n      error:\n        type: string\n        nullable: true"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"total\": 1, \"entries\": [{\"name\": \"tiny-random-VideoMAEForVideoCl...\", \"enabled\": true, \"weight\": 0.53}], \"details\": {\"arg_count\": 0, \"requirements\": [], \"id\": 23147, \"error\": null, \"latency_ms\": 128.56}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      arg_count:\n        type: integer\n      requirements:\n        type: array\n        items: {}\n      id:\n        type: integer\n      error:\n        type: string\n        nullable: true\n      latency_ms:\n        type: number"}
{"input_json": "{\"service\": \"mrm8488/bert-multi-cased-finet...\", \"config\": {\"domain\": \"Natural Language Processi...\", \"score\": 0.823, \"latency_ms\": 66.82, \"error\": null, \"status_code\": 404, \"arg_count\": 0}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      domain:\n        type: string\n      score:\n        type: number\n      latency_ms:\n        type: number\n      error:\n        type: string\n        nullable: true\n      status_code:\n        type: integer\n      arg_count:\n        type: integer\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"service\": \"d4data/Indian-voice-cloning\", \"config\": {\"requirements\": [], \"arguments\": [], \"arg_count\": 0, \"deprecated\": true, \"error\": null}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      requirements:\n        type: array\n        items: {}\n      arguments:\n        type: array\n        items: {}\n      arg_count:\n        type: integer\n      deprecated:\n        type: boolean\n      error:\n        type: string\n        nullable: true\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"provider\": \"TensorFlow Hub\", \"total\": 1, \"entries\": [{\"name\": \"mobilenet_v2\", \"enabled\": true, \"weight\": 0.66}], \"details\": {\"endpoint\": \"tf.keras.Sequential([hub.KerasLayer('htt...\", \"arguments\": [], \"framework\": \"TensorFlow Hub\", \"score\": 0.737}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      endpoint:\n        type: string\n      arguments:\n        type: array\n        items: {}\n      framework:\n        type: string\n      score:\n        type: number"}
{"input_json": "{\"provider\": \"TensorFlow Hub\", \"total\": 2, \"entries\": [{\"name\": \"model_id\", \"enabled\": true, \"weight\": 0.81}], \"details\": {\"active\": false, \"api_name\": \"model_id\", \"score\": 0.658, \"requirements\": [\"tensorflow\", \"tensorflow_hub\"], \"tags\": [\"Image object detection\", \"TensorFlow Hub\"], \"status_code\": 202}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      active:\n        type: boolean\n      api_name:\n        type: string\n      score:\n        type: number\n      requirements:\n        type: array\n        items:\n          type: string\n      tags:\n        type: array\n        items:\n          type: string\n      status_code:\n        type: integer"}
{"input_json": "{\"domain\": \"Natural Language Processi...\", \"requirements\": [\"transformers\"], \"latency_ms\": 367.47, \"provider\": \"Hugging Face Transformers\", \"endpoint\": \"pipeline('text2text-generation', 'pszemr...\", \"active\": true}", "target_yaml": "type: object\nproperties:\n  domain:\n    type: string\n  requirements:\n    type: array\n    items:\n      type: string\n  latency_ms:\n    type: number\n  provider:\n    type: string\n  endpoint:\n    type: string\n  active:\n    type: boolean"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"total\": 1, \"entries\": [{\"name\": \"hf-tiny-model-private/tiny-ran...\", \"enabled\": true, \"weight\": 0.8}], \"details\": {\"active\": false, \"latency_ms\": 215.54, \"requirements\": [\"transformers\"]}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      active:\n        type: boolean\n      latency_ms:\n        type: number\n      requirements:\n        type: array\n        items:\n          type: string"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"provider\": \"Flair\", \"arguments\": [\"sentence\"], \"score\": 0.828}, \"meta\": {\"version\": 2, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      provider:\n        type: string\n      arguments:\n        type: array\n        items:\n          type: string\n      score:\n        type: number\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"service\": \"universal-sentence-encoder\", \"config\": {\"score\": 0.819, \"id\": 24641, \"provider\": \"TensorFlow Hub\", \"arg_count\": 0, \"deprecated\": false, \"status_code\": 201}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      score:\n        type: number\n      id:\n        type: integer\n      provider:\n        type: string\n      arg_count:\n        type: integer\n      deprecated:\n        type: boolean\n      status_code:\n        type: integer\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"score\": 0.934, \"framework\": \"Transformers\", \"deprecated\": false}", "target_yaml": "type: object\nproperties:\n  score:\n    type: number\n  framework:\n    type: string\n  deprecated:\n    type: boolean"}
{"input_json": "{\"service\": \"universal-sentence-encoder\", \"config\": {\"arg_count\": 0, \"id\": 67599, \"api_name\": \"universal-sentence-encoder\", \"tags\": [\"Text embedding\", \"TensorFlow Hub\"]}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      arg_count:\n        type: integer\n      id:\n        type: integer\n      api_name:\n        type: string\n      tags:\n        type: array\n        items:\n          type: string\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"endpoint\": \"AutoModelForCausalLM.from_pretrained('mi...\", \"arguments\": [], \"id\": 88233, \"deprecated\": false, \"arg_count\": 0, \"score\": 0.516}, \"meta\": {\"version\": 3, \"cached\": true}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      endpoint:\n        type: string\n      arguments:\n        type: array\n        items: {}\n      id:\n        type: integer\n      deprecated:\n        type: boolean\n      arg_count:\n        type: integer\n      score:\n        type: number\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"provider\": \"TensorFlow Hub\", \"total\": 2, \"entries\": [{\"name\": \"model_id\", \"enabled\": true, \"weight\": 0.59}], \"details\": {\"id\": 93161, \"tags\": [\"Image feature vector\", \"TensorFlow Hub\"], \"framework\": \"TensorFlow Hub\", \"status_code\": 400, \"domain\": \"Image feature vector\", \"requirements\": [\"tensorflow\", \"tensorflow_hub\"]}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      id:\n        type: integer\n      tags:\n        type: array\n        items:\n          type: string\n      framework:\n        type: string\n      status_code:\n        type: integer\n      domain:\n        type: string\n      requirements:\n        type: array\n        items:\n          type: string"}
{"input_json": "{\"latency_ms\": 50.98, \"deprecated\": true, \"requirements\": [\"transformers\", \"torch\", \"datasets\"], \"domain\": \"Computer Vision Image Cla...\"}", "target_yaml": "type: object\nproperties:\n  latency_ms:\n    type: number\n  deprecated:\n    type: boolean\n  requirements:\n    type: array\n    items:\n      type: string\n  domain:\n    type: string"}
{"input_json": "{\"status\": \"ok\", \"data\": {\"framework\": \"TensorFlow Hub\", \"arg_count\": 1, \"latency_ms\": 397.19, \"domain\": \"Image object detection\"}, \"meta\": {\"version\": 3, \"cached\": false}}", "target_yaml": "type: object\nproperties:\n  status:\n    type: string\n  data:\n    type: object\n    properties:\n      framework:\n        type: string\n      arg_count:\n        type: integer\n      latency_ms:\n        type: number\n      domain:\n        type: string\n  meta:\n    type: object\n    properties:\n      version:\n        type: integer\n      cached:\n        type: boolean"}
{"input_json": "{\"endpoint\": \"torch.hub.load(repo_or_dir='ultralytics/...\", \"score\": 0.54, \"framework\": \"PyTorch\", \"latency_ms\": 148.94}", "target_yaml": "type: object\nproperties:\n  endpoint:\n    type: string\n  score:\n    type: number\n  framework:\n    type: string\n  latency_ms:\n    type: number"}
{"input_json": "{\"endpoint\": \"hub.KerasLayer('https://tfhub.dev/google...\", \"error\": null, \"deprecated\": true, \"arg_count\": 0}", "target_yaml": "type: object\nproperties:\n  endpoint:\n    type: string\n  error:\n    type: string\n    nullable: true\n  deprecated:\n    type: boolean\n  arg_count:\n    type: integer"}
{"input_json": "{\"service\": \"Salesforce/blip-vqa-capfilt-la...\", \"config\": {\"id\": 42829, \"tags\": [\"Multimodal Visual Questio...\", \"Hugging Face Transformers\"], \"error\": null, \"score\": 0.92}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      id:\n        type: integer\n      tags:\n        type: array\n        items:\n          type: string\n      error:\n        type: string\n        nullable: true\n      score:\n        type: number\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"service\": \"universal-sentence-encoder\", \"config\": {\"error\": null, \"deprecated\": true, \"framework\": \"TensorFlow Hub\", \"id\": 47167, \"arg_count\": 1}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      error:\n        type: string\n        nullable: true\n      deprecated:\n        type: boolean\n      framework:\n        type: string\n      id:\n        type: integer\n      arg_count:\n        type: integer\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"service\": \"csarron/bert-base-uncased-squa...\", \"config\": {\"status_code\": 201, \"framework\": \"Hugging Face Transformers\", \"latency_ms\": 183.7, \"active\": false}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      status_code:\n        type: integer\n      framework:\n        type: string\n      latency_ms:\n        type: number\n      active:\n        type: boolean\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"arg_count\": 0, \"score\": 0.579, \"api_name\": \"ydshieh/vit-gpt2-coco-en\", \"provider\": \"Hugging Face Transformers\", \"active\": false}", "target_yaml": "type: object\nproperties:\n  arg_count:\n    type: integer\n  score:\n    type: number\n  api_name:\n    type: string\n  provider:\n    type: string\n  active:\n    type: boolean"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"arg_count\": 0, \"error\": null, \"status_code\": 200, \"deprecated\": false, \"id\": 85832}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  arg_count:\n    type: integer\n  error:\n    type: string\n    nullable: true\n  status_code:\n    type: integer\n  deprecated:\n    type: boolean\n  id:\n    type: integer"}
{"input_json": "{\"service\": \"tuner007/pegasus_paraphrase\", \"config\": {\"endpoint\": \"PegasusForConditionalGeneration.from_pre...\", \"id\": 46592, \"api_name\": \"tuner007/pegasus_paraphrase\", \"deprecated\": true, \"provider\": \"Hugging Face Transformers\", \"score\": 0.59}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      endpoint:\n        type: string\n      id:\n        type: integer\n      api_name:\n        type: string\n      deprecated:\n        type: boolean\n      provider:\n        type: string\n      score:\n        type: number\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"provider\": \"TensorFlow Hub\", \"total\": 1, \"entries\": [{\"name\": \"deeplabv3_257_mv_gpu\", \"enabled\": true, \"weight\": 0.92}], \"details\": {\"arguments\": [], \"requirements\": [], \"score\": 0.863, \"arg_count\": 0, \"status_code\": 202, \"latency_ms\": 433.58}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      arguments:\n        type: array\n        items: {}\n      requirements:\n        type: array\n        items: {}\n      score:\n        type: number\n      arg_count:\n        type: integer\n      status_code:\n        type: integer\n      latency_ms:\n        type: number"}
{"input_json": "{\"provider\": \"Hugging Face Transformers\", \"total\": 1, \"entries\": [{\"name\": \"glpn-nyu-finetuned-diode-22121...\", \"enabled\": true, \"weight\": 0.13}], \"details\": {\"api_name\": \"glpn-nyu-finetuned-diode-22121...\", \"requirements\": [], \"tags\": [\"Computer Vision Depth Est...\", \"Hugging Face Transformers\"]}}", "target_yaml": "type: object\nproperties:\n  provider:\n    type: string\n  total:\n    type: integer\n  entries:\n    type: array\n    items:\n      type: object\n      properties:\n        name:\n          type: string\n        enabled:\n          type: boolean\n        weight:\n          type: number\n  details:\n    type: object\n    properties:\n      api_name:\n        type: string\n      requirements:\n        type: array\n        items: {}\n      tags:\n        type: array\n        items:\n          type: string"}
{"input_json": "{\"service\": \"imagenet_mobilenet_v2_100_224_...\", \"config\": {\"arguments\": [], \"tags\": [], \"provider\": \"TensorFlow Hub\", \"api_name\": \"imagenet_mobilenet_v2_100_224_...\", \"status_code\": 200}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      arguments:\n        type: array\n        items: {}\n      tags:\n        type: array\n        items: {}\n      provider:\n        type: string\n      api_name:\n        type: string\n      status_code:\n        type: integer\n  warnings:\n    type: array\n    items: {}"}
{"input_json": "{\"tags\": [\"Reinforcement Learning\", \"Hugging Face Transformers\"], \"arg_count\": 0, \"requirements\": [\"transformers\"], \"active\": true}", "target_yaml": "type: object\nproperties:\n  tags:\n    type: array\n    items:\n      type: string\n  arg_count:\n    type: integer\n  requirements:\n    type: array\n    items:\n      type: string\n  active:\n    type: boolean"}
{"input_json": "{\"latency_ms\": 192.59, \"provider\": \"TensorFlow Hub\", \"endpoint\": \"hub.load('https://tfhub.dev/google/moven...\"}", "target_yaml": "type: object\nproperties:\n  latency_ms:\n    type: number\n  provider:\n    type: string\n  endpoint:\n    type: string"}
{"input_json": "{\"requirements\": [], \"tags\": [], \"score\": 0.7, \"deprecated\": true}", "target_yaml": "type: object\nproperties:\n  requirements:\n    type: array\n    items: {}\n  tags:\n    type: array\n    items: {}\n  score:\n    type: number\n  deprecated:\n    type: boolean"}
{"input_json": "{\"service\": \"vennify/t5-base-grammar-correc...\", \"config\": {\"status_code\": 201, \"endpoint\": \"HappyTextToText('T5', 'vennify/t5-base-g...\", \"active\": true}, \"warnings\": []}", "target_yaml": "type: object\nproperties:\n  service:\n    type: string\n  config:\n    type: object\n    properties:\n      status_code:\n        type: integer\n      endpoint:\n        type: string\n      active:\n        type: boolean\n  warnings:\n    type: array\n    items: {}"}

In [ ]:
!wc -l /content/data/test.jsonl

## 5. Run the evaluation on all 100 samples

Expect roughly 30–60 min on a T4.

In [ ]:
%cd /content
!python evaluate_ollama.py --limit 100

## 6. Show summary and download results

In [ ]:
import json
with open("/content/results/ollama_summary.json") as f:
    summary = json.load(f)

import pandas as pd
pd.DataFrame(summary).T

In [ ]:
#!cd /content && zip -qr results.zip results
#from google.colab import files
#files.download("/content/results.zip")

### (Optional) Save results to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/json2openapi-llm
# !cp -r /content/results /content/drive/MyDrive/json2openapi-llm/